## Thực hành 1a


In [ ]:
# Ham tong quat tinh tich phan nD chieu

import random as random
import numpy as np
import time

def tich_phan_monte_carlo_nhieu_chieu(a,b,N,N_D,hamso):
    S = 0
    for i in range(N): #1 diem ngau nhien se bao gom N diem la so chieu chieu D
        x = []
        for j in range(N_D):   #Dung de tao x1 x2 x3...xD ngau nhien sau do nhet vo hamso(x)
            x.append(np.random.uniform(a[j],b[j]))
        S = S + hamso(x)
    # Phan tinh tich cuoi cung de ra ket qua
    tich = 1
    for j in range(N_D):
        tich = tich * (b[j]-a[j])
    ketqua = tich * S/N
    return ketqua

def hamso_Nchieu(x): #x1^2 +x^2 +...
    a = 0
    for j in range(len(x)): #len x la so chieu
        a = a + x[j]**2 #dung de tinh ham so a = x1^2 +...
    return a

In [ ]:
N_D = [3,4,10]

a = [[0]*d for d in N_D] #day la so chieu
b = [[1]*d for d in N_D]

# So luong diem ngau nhien
N = 100000

with open(f"MC_1A_ketqua", "w", encoding="utf-8") as file:
    file.write(f"Ket qua tinh tich phan MC va thoi gian tinh cho truong hop nD\n")
    file.write(f"#" * 80 + "\n" + "\n")

    for i in range(len(N_D)):

        t_start = time.perf_counter()

        ketqua_monte_carlo  = tich_phan_monte_carlo_nhieu_chieu(a[i],b[i],N,N_D[i],hamso_Nchieu)

        t_end = time.perf_counter()

        thoigianchay = t_end-t_start

        file.write(f"Ket qua chay tinh tich phan {N_D[i]}D la: {ketqua_monte_carlo:.12f}\n\n")
        file.write(f"Thoi gian chay tinh bang Monte Carlo la: {thoigianchay:.12f} s\n\n")
        file.write("#" * 80 + "\n\n")

### Phan 2: Uoc luong sai so 


In [3]:
def MC_nhieuchieu_tinh_saiso(a, b, N, N_D, hamso):
    tong_f = 0.0
    tong_f2 = 0.0

    for i in range(N):
        x = []
        for j in range(N_D):
            x.append(np.random.uniform(a[j], b[j]))

        fx = hamso(x)
        tong_f += fx
        tong_f2 += fx**2

    V = 1.0
    for j in range(N_D):
        V *= (b[j] - a[j])

    f_tb = tong_f / N
    ketqua = V * f_tb

    if N > 1:
        phuong_sai_mau = (tong_f2 - N * f_tb**2) / (N - 1)
        sai_so_uoc_luong = V * np.sqrt(phuong_sai_mau / N)
    else:
        sai_so_uoc_luong = 0.0

    return ketqua, sai_so_uoc_luong

In [7]:
N_D = [3,4]

a = [[0]*d for d in N_D]
b = [[1]*d for d in N_D]

nguong_saiso = 1e-4
N_max = int(1/nguong_saiso**2)

with open("MC_1A_saiso.txt", "w", encoding="utf-8") as file:
    file.write("Ket qua tinh tich phan MC cho truong hop nD\n")
    file.write("#" * 80 + "\n\n")
    file.write(f"{'N_D':>10s} {'N':>12s} {'ket_qua':>16s} {'sai_so':>16s}\n")

    for i in range(len(N_D)):
        da_tim_thay = False
        
        N = 1000

        while N <= N_max:
            ketqua_monte_carlo, sai_so = MC_nhieuchieu_tinh_saiso(a[i], b[i], N, N_D[i], hamso_Nchieu)
            file.write(f"{N_D[i]:10d} {N:12d} {ketqua_monte_carlo:16.9f} {sai_so:16.9f}\n")

            if sai_so < nguong_saiso:
                file.write(f"Da tim thay N toi uu cho tich phan MC {N_D[i]:10d} chieu: N = {N}\n")
                file.write("#" * 80 + "\n\n")
                da_tim_thay = True
                break

            N = N * 2 # Neu dung buoc nhay N thi chay lau

        if da_tim_thay == False:
            file.write(f"{N_D[i]:10d} {'khong tim thay':>12s} {'-':>16s} {'-':>16s}\n")
            file.write("#" * 80 + "\n\n")

KeyboardInterrupt: 

## Thực hành 1b

In [3]:
def hamso_Nchieu_1B(x):
    a = 0
    for j in range(len(x)):
        a = a + x[j]
    return a**2

def tinhtongRiemann(a,b,N,N_D,hamso):
    delta_h = []
    for j in range(len(a)):
        delta_h.append((b[j]-a[j])/N)
    dV = 1
    for j in range(len(a)):
        dV = dV* delta_h[j]
    
    S = 0.0
    x = [0.0] * N_D

    def de_quy(chieu):
        nonlocal S

        if chieu == N_D:
            S += hamso(x)
            return

        for i in range(N):
            x[chieu] = a[chieu] + i * delta_h[chieu]
            de_quy(chieu + 1)

    de_quy(0)

    return S * dV

def tich_phan_monte_carlo_nhieu_chieu(a, b, N, N_D, hamso):
    S = 0.0
    for i in range(N):
        x = []
        for j in range(N_D):
            x.append(np.random.uniform(a[j], b[j]))
        S += hamso(x)

    tich = 1.0
    for j in range(N_D):
        tich *= (b[j] - a[j])

    ketqua = tich * S / N
    return ketqua

N_RM = 6
N_MC = 10000

N_D = 10
a = [0] * N_D
b = [1] * N_D


t_start_RM = time.perf_counter()
tong_Riemann = tinhtongRiemann(a, b, N_RM, N_D, hamso_Nchieu_1B)
t_end_RM = time.perf_counter()

thoigianchay_Riemann = t_end_RM - t_start_RM


t_start_MC = time.perf_counter()
ketqua_monte_carlo = tich_phan_monte_carlo_nhieu_chieu(a, b, N_MC, N_D, hamso_Nchieu_1B)
t_end_MC = time.perf_counter()

thoigianchay_MC = t_end_MC - t_start_MC

nghiem_calculus = 155/6

delta_MC = np.abs(ketqua_monte_carlo - nghiem_calculus)
delta_RM = np.abs(tong_Riemann - nghiem_calculus)

with open("MC_1B_thoigianchay.txt", "w", encoding="utf-8") as file:
    file.write("Ket qua tinh tich phan bang phuong phap Monte Carlo va tong Riemann\n")
    file.write("#" * 80 + "\n\n")

    file.write(f"Ket qua tinh tich phan bang tong Riemann la: {tong_Riemann:.12f}\n")
    file.write(f"Thoi gian chay bang tong Riemann la: {thoigianchay_Riemann:.12f} s\n")
    file.write(f"Sai so so voi nghiem giai tich bang tong Riemann la: {delta_RM:.12f}\n\n")

    file.write("#" * 80 + "\n\n")

    file.write(f"Ket qua tinh tich phan bang Monte Carlo la: {ketqua_monte_carlo:.12f}\n")
    file.write(f"Thoi gian chay bang Monte Carlo la: {thoigianchay_MC:.12f} s\n")
    file.write(f"Sai so so voi nghiem giai tich bang Monte Carlo la: {delta_MC:.12f}\n")

## Thực hành MC 2

In [ ]:
def hamso_Nchieu_MC2_A(x):
    a = 0
    for j in range(len(x)):
        a = a + x[j]**2
    return a

N_D = 3
a = [0]*N_D
b = [1]*N_D
N = 1000000

ketqua_MC2, saiso_uocluong_MC2 = MC_nhieuchieu_tinh_saiso(a, b, N, N_D, hamso_Nchieu_MC2_A)

with open("MC_2A.txt", "w", encoding="utf-8") as file:
    file.write("Ket qua tinh tich phan bang phuong phap Monte Carlo\n")
    file.write("#" * 80 + "\n\n")
    file.write(f"So luong diem ngau nhien la: {int(N):8d}\n")
    file.write(f"Ket qua tinh tich phan bang Monte Carlo la: {ketqua_MC2:.12f}\n")
    file.write(f"Sai so so uoc luong Monte Carlo la: {saiso_uocluong_MC2:.12f}\n")

In [15]:
def hamso_Nchieu_MC2_A(x):
    if x[0]**2 + x[1]**2 + x[2]**2 <= 1 and x[0]**2 + x[1]**2 <= 0.25:
        return 1
    else:
        return 0

def monte_carlo_the_tich(N):
    dem = 0
    for i in range(N):
        x = np.random.uniform(-1, 1)
        y = np.random.uniform(-1, 1)
        z = np.random.uniform(-1, 1)

        if hamso_Nchieu_MC2_A([x, y, z]) == 1:
            dem += 1

    V_hop = 8
    V_uocluong = V_hop * dem / N
    return V_uocluong

N = 1000000

the_tich  = monte_carlo_the_tich(N)

with open("MC_2B.txt", "w", encoding="utf-8") as file:
    file.write("Ket qua tinh the tich bang tich phan theo phuong phap Monte Carlo\n")
    file.write("#" * 80 + "\n\n")
    file.write(f"So luong diem ngau nhien la: {int(N):8d}\n")
    file.write(f"Ket qua cua the tich giao diem la: {the_tich:.9f}\n")
